[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C20_Frontier_Architectures_Course/02_attention_variants/02_attention_variants.ipynb)

# 02 · 注意力变体 MHA→GQA→MLA（从零实现）

目标：实现 MHA、GQA(repeat_kv)、MLA(低秩压缩)，并**算清各自的 KV cache 字节账**。

路线：softmax → MHA → KV cache 账 → GQA(repeat_kv) → MLA 低秩 → ✏️ 练习 → 🧪 真实模型胶囊。

## 1 · 基础件：softmax 与因果注意力

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def causal_mask(T):
    return np.triu(np.ones((T, T), dtype=bool), k=1)

def sdpa(Q, K, V):
    # 单头缩放点积注意力 Q,K,V: (T, d_h)
    d_h = Q.shape[-1]
    s = Q @ K.T / np.sqrt(d_h)
    s = np.where(causal_mask(Q.shape[0]), -np.inf, s)
    return softmax(s, -1) @ V

T, d_h = 6, 8
Q = rng.standard_normal((T, d_h))
out = sdpa(Q, Q, Q)
print('单头注意力输出形状', out.shape)
assert out.shape == (T, d_h)

## 2 · MHA：多头注意力

把 d_model 切成 h 个头，每头独立算注意力再拼接、过输出投影。

In [ ]:
def split_heads(x, h):
    T, d = x.shape
    return x.reshape(T, h, d // h).transpose(1, 0, 2)   # (h, T, d_h)

def merge_heads(x):
    h, T, dh = x.shape
    return x.transpose(1, 0, 2).reshape(T, h * dh)

def mha(X, Wq, Wk, Wv, Wo, h):
    Q, Kk, Vv = X @ Wq, X @ Wk, X @ Wv
    Qh, Kh, Vh = split_heads(Q, h), split_heads(Kk, h), split_heads(Vv, h)
    out = np.stack([sdpa(Qh[i], Kh[i], Vh[i]) for i in range(h)])  # (h,T,dh)
    return merge_heads(out) @ Wo

d_model, h = 32, 4
X = rng.standard_normal((T, d_model))
W = [rng.standard_normal((d_model, d_model)) / np.sqrt(d_model) for _ in range(4)]
print('MHA 输出形状', mha(X, *W, h).shape, '（应为 (T, d_model)）')
assert mha(X, *W, h).shape == (T, d_model)

## 3 · KV cache 字节账

`KV = 2 · L · T · n_kv_heads · d_h · b`。把 MHA / MQA / GQA 放一起比。

In [ ]:
def kv_cache_bytes(L, T, n_kv_heads, d_h, b=2):
    return 2 * L * T * n_kv_heads * d_h * b

# Llama-2-7B 量级：L=32, h=32, d_h=128, d_model=4096, fp16, T=4096
L, h_, d_h_, Tctx = 32, 32, 128, 4096
for name, n_kv in [('MHA', 32), ('GQA g=8', 8), ('MQA', 1)]:
    gb = kv_cache_bytes(L, Tctx, n_kv, d_h_) / 1e9
    print(f'{name:9s} n_kv={n_kv:2d} -> KV cache {gb:6.3f} GB/序列')
assert kv_cache_bytes(L,Tctx,8,d_h_) * 4 == kv_cache_bytes(L,Tctx,32,d_h_)
print('\n结论：GQA(8) 把 4GB 砍到 1GB，MQA 砍到 0.13GB。')

## 4 · GQA 与 repeat_kv

q 头 h 个、kv 头 g 个，`repeat_kv` 把每份 K/V 复制 `h/g` 次扩回 h 头。

In [ ]:
def repeat_kv(kv, n_rep):
    # kv: (g, T, d_h) -> (g*n_rep, T, d_h)
    return np.repeat(kv, n_rep, axis=0)

def gqa(X, Wq, Wk, Wv, Wo, h, g):
    d = X.shape[1]; d_h = d // h
    Q = split_heads(X @ Wq, h)                     # (h, T, d_h)
    Kg = split_heads(X @ Wk[:, :g*d_h], g)         # (g, T, d_h)
    Vg = split_heads(X @ Wv[:, :g*d_h], g)
    Kh, Vh = repeat_kv(Kg, h // g), repeat_kv(Vg, h // g)
    out = np.stack([sdpa(Q[i], Kh[i], Vh[i]) for i in range(h)])
    return merge_heads(out) @ Wo

out_gqa = gqa(X, *W, h=4, g=2)
print('GQA 输出形状', out_gqa.shape)
print('g=h 时 GQA 应等价 MHA：', np.allclose(gqa(X, *W, h=4, g=4), mha(X, *W, 4)))
assert np.allclose(gqa(X, *W, h=4, g=4), mha(X, *W, 4))

## 5 · MLA：低秩潜向量压缩 KV

下投影到潜向量 `c`（维度 d_c），**只缓存 c**；用时上投影回 K/V。缓存量从 `2·d_model` 降到 `d_c`。

In [ ]:
def mla(X, W_DKV, W_UK, W_UV, Wq, Wo, h):
    d = X.shape[1]
    c = X @ W_DKV                 # (T, d_c)  <- 只有它进 KV cache
    Kk = c @ W_UK                 # (T, d)    用时展开
    Vv = c @ W_UV
    Q = split_heads(X @ Wq, h); Kh = split_heads(Kk, h); Vh = split_heads(Vv, h)
    out = np.stack([sdpa(Q[i], Kh[i], Vh[i]) for i in range(h)])
    return merge_heads(out) @ Wo, c

d_c = 8                          # 远小于 d_model=32
W_DKV = rng.standard_normal((d_model, d_c)) / np.sqrt(d_model)
W_UK  = rng.standard_normal((d_c, d_model)) / np.sqrt(d_c)
W_UV  = rng.standard_normal((d_c, d_model)) / np.sqrt(d_c)
out_mla, c = mla(X, W_DKV, W_UK, W_UV, W[0], W[3], h)
print('MLA 输出形状', out_mla.shape, '| 每 token 缓存维度 d_c =', c.shape[1], '（MHA 需缓存 2*d_model =', 2*d_model, '）')
assert out_mla.shape == (T, d_model) and c.shape[1] == d_c

> MLA 每 token 只缓存 `d_c=8`，而 MHA 要缓存 `2·d_model=64`——**省 8×**，且 K/V 仍是满秩多头还原，质量接近 MHA。

---
## ✏️ 练习 1：KV cache 缩小倍数

实现 `kv_savings(n_kv_mha, n_kv_gqa)` 返回 MHA 相对 GQA 的 KV cache 倍数。

In [ ]:
def kv_savings(n_kv_mha, n_kv_gqa):
    # TODO: 返回 MHA 的 KV cache 是 GQA 的多少倍（其余参数相同）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert kv_savings(32, 8) == 4
assert kv_savings(32, 1) == 32
assert abs(kv_savings(32, 32) - 1) < 1e-9
print('✅ 练习 1 通过')

## ✏️ 练习 2：实现 repeat_kv

用你的 repeat_kv 把 `(g,T,d_h)` 扩成 `(h,T,d_h)`，验证复制正确。

In [ ]:
def my_repeat_kv(kv, n_rep):
    # TODO: kv (g,T,d_h) -> (g*n_rep, T, d_h)，沿 head 维重复
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
kv = rng.standard_normal((2, 5, 4))
r = my_repeat_kv(kv, 3)
assert r.shape == (6, 5, 4)
assert np.allclose(r[0], kv[0]) and np.allclose(r[2], kv[0]) and np.allclose(r[3], kv[1])
print('✅ 练习 2 通过')

## ✏️ 练习 3：MLA 上/下投影的低秩重构

给定满秩 K，用 SVD 取秩 `d_c` 近似，验证重构误差随 `d_c` 增大而下降。

In [ ]:
def low_rank_reconstruct(Kfull, d_c):
    # TODO: 对 Kfull (T, d) 做 SVD，保留前 d_c 个奇异值，返回重构矩阵 (T, d)
    # 提示：U,S,Vt = np.linalg.svd(Kfull, full_matrices=False); 截断前 d_c
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Kfull = rng.standard_normal((20, 16))
err = [np.linalg.norm(Kfull - low_rank_reconstruct(Kfull, r)) for r in [2, 4, 8, 16]]
assert all(err[i] >= err[i+1] - 1e-9 for i in range(len(err)-1)), '秩越大误差越小'
assert err[-1] < 1e-8, '秩=满秩应几乎无损'
print('✅ 练习 3 通过 | 重构误差', [round(e,3) for e in err])

---
### 📖 参考答案

In [ ]:
# 练习 1
def kv_savings(n_kv_mha, n_kv_gqa):
    return n_kv_mha / n_kv_gqa

# 练习 2
def my_repeat_kv(kv, n_rep):
    return np.repeat(kv, n_rep, axis=0)

# 练习 3
def low_rank_reconstruct(Kfull, d_c):
    U, S, Vt = np.linalg.svd(Kfull, full_matrices=False)
    return (U[:, :d_c] * S[:d_c]) @ Vt[:d_c]

---
## 🧪 真实数据胶囊：真实模型的 KV cache 对比

用真实模型配置算 8K 上下文下每序列的 KV cache，体会 GQA/MLA 的威力。

In [ ]:
# 取自公开技术报告/配置
MODELS = {
    'GPT-3-175B (MHA)':   dict(L=96, h=96, n_kv=96, d_h=128),
    'Llama-2-70B (GQA8)': dict(L=80, h=64, n_kv=8,  d_h=128),
    'Llama-3-8B (GQA8)':  dict(L=32, h=32, n_kv=8,  d_h=128),
}
Tctx2 = 8192
for name, m in MODELS.items():
    gb = kv_cache_bytes(m['L'], Tctx2, m['n_kv'], m['d_h']) / 1e9
    print(f'{name:22s} -> {gb:6.2f} GB/序列 @8K')
print('\n注意：70B 用 GQA8 后 KV cache 反而比 8B 的 MHA 等价物更可控——GQA 是大模型长上下文的必需品。')

**🧪 胶囊练习**：实现 `mla_cache_ratio(d_model, d_c)`，返回 MLA 相对 MHA 每 token 的缓存压缩比 `2*d_model/d_c`。

In [ ]:
def mla_cache_ratio(d_model, d_c):
    # TODO
    raise NotImplementedError

In [ ]:
# 自测
assert abs(mla_cache_ratio(5120, 512) - 20.0) < 1e-9
print('✅ DeepSeek-V2 量级：MLA 每 token 缓存约为 MHA 的 1/20')

In [ ]:
# 📖 胶囊参考答案
def mla_cache_ratio(d_model, d_c):
    return 2 * d_model / d_c

---
### 小结
- 解码瓶颈是显存带宽 → 目标是压 KV cache。
- MHA→MQA→GQA→MLA：逐步压缩，GQA 是当前甜点，MLA 最省。
- repeat_kv 是 GQA 的工程核心；MLA 用低秩潜向量 + 解耦 RoPE。

下一站：**模块 03 · MoE 与 SwiGLU**。